In [0]:
-- ============================================================
-- VALIDATION TEST:
-- Find columns that exist in warehouse tables
-- but are missing from metadata.data_dictionary
-- ============================================================

WITH actual_columns AS (

    SELECT
        table_schema AS schema_name,
        table_name,
        column_name,
        data_type
    FROM system.information_schema.columns
    WHERE table_schema IN ('gold', 'silver', 'bronze')

),

dictionary_columns AS (

    SELECT
        schema_name,
        table_name,
        column_name,
        data_type
    FROM metadata.data_dictionary

)

SELECT
    a.schema_name,
    a.table_name,
    a.column_name,
    a.data_type AS actual_data_type,
    d.data_type AS dictionary_data_type,

    CASE
        WHEN d.column_name IS NULL
            THEN 'MISSING FROM DATA DICTIONARY'

        WHEN
            CASE
                WHEN UPPER(a.data_type) = 'LONG' THEN 'BIGINT'
                ELSE UPPER(a.data_type)
            END
            <>
            CASE
                WHEN UPPER(d.data_type) = 'LONG' THEN 'BIGINT'
                ELSE UPPER(d.data_type)
            END
            THEN 'DATA TYPE MISMATCH'

        ELSE 'VALID'
    END AS validation_status

FROM actual_columns a

LEFT JOIN dictionary_columns d
    ON  a.schema_name = d.schema_name
    AND a.table_name = d.table_name
    AND a.column_name = d.column_name

WHERE
    d.column_name IS NULL
    OR UPPER(a.data_type) <> UPPER(d.data_type)

ORDER BY
    a.schema_name,
    a.table_name,
    a.column_name;